# Simulation Trace Pattern Mining

Load and parse the simulation trace log, then explore patterns.

In [ ]:
import sys
sys.path.insert(0, '..')

from analysis.trace_miner import parse_trace, enrich_forward_returns, verify_parser, build_sequences
from analysis.pattern_miner import run_tier1, run_tier2
import pandas as pd
import numpy as np

TRACE_PATH = '../../logs/backend/simulation_trace.log'

events_df, candles_df, line_prices = parse_trace(TRACE_PATH)
events_df = enrich_forward_returns(events_df, candles_df)

print(f'Events: {len(events_df)} | Candles: {len(candles_df)} | Line price keys: {len(line_prices)}')
print(f'\nEvent type distribution:\n{events_df["event_type"].value_counts()}')

In [ ]:
verify_result = verify_parser(events_df, candles_df, TRACE_PATH)
print('=' * 60)
print('VERIFICATION REPORT')
print('=' * 60)
for check in verify_result['checks']:
    status = 'PASS' if check['passed'] else 'FAIL'
    print(f'[{status}] {check["name"]}: {check["detail"]}')
print(f'\nOverall: {"ALL CHECKS PASSED" if verify_result["passed"] else "SOME CHECKS FAILED"}')

In [ ]:
tier1_results = run_tier1(events_df)

if len(tier1_results) > 0:
    print(f'Tier 1 candidates: {len(tier1_results)}')
    display(tier1_results[['pattern', 'sample_count', 'mean_mfe_10', 'mean_mae_10', 'win_rate', 'composite']].head(20))
else:
    print('No patterns passed Tier 1 thresholds.')
    print(f'\nThresholds: sample >= 20, win_rate >= 0.50')

In [ ]:
et_filter = 'SUPPORT_TEST'
frac_filter = '0.5'
cp_filter = ''  # leave empty for any

non_retro = events_df[~events_df['is_retro']]
mask = (non_retro['event_type'] == et_filter) & (non_retro['line_fraction'] == str(frac_filter))
if cp_filter:
    mask = mask & (non_retro['candle_pattern'] == cp_filter)

subset = non_retro[mask]
print(f'Pattern: {et_filter} on {frac_filter} line' + (f' [{cp_filter}]' if cp_filter else ''))
print(f'Occurrences: {len(subset)}')
if len(subset) > 0:
    print(f'Mean MFE 10: {subset["fwd_mfe_10"].mean():.2f}%')
    print(f'Mean MAE 10: {subset["fwd_mae_10"].mean():.2f}%')
    print(f'Win Rate 10: {subset["fwd_win_10"].astype(bool).mean():.2%}')
    print(f'\nSample events:')
    display(subset[['bar_index', 'timestamp', 'close', 'candle_pattern', 'fwd_mfe_10', 'fwd_win_10']].head(10))

In [ ]:
candles_df['ema20'] = candles_df['close'].ewm(span=20, adjust=False).mean()
candles_df['trend_slope'] = candles_df['ema20'].diff(5) / candles_df['close'] * 100

events_df = events_df.merge(
    candles_df[['bar_index', 'trend_slope']],
    on='bar_index', how='left'
)

uptrend = events_df['trend_slope'] > 0.1
downtrend = events_df['trend_slope'] < -0.1
flat = ~uptrend & ~downtrend

for label, mask in [('UPTREND', uptrend), ('DOWNTREND', downtrend), ('FLAT', flat)]:
    subset = events_df[mask & (events_df['event_type'] == 'SUPPORT_TEST') & (events_df['line_fraction'] == '0.5')]
    if len(subset) > 0:
        print(f'{label}: n={len(subset)}, MFE_10={subset["fwd_mfe_10"].mean():.2f}%, Win={subset["fwd_win_10"].astype(bool).mean():.2%}')

In [ ]:
import matplotlib.pyplot as plt

et = 'SUPPORT_TEST'
frac = '0.5'
subset = events_df[(events_df['event_type'] == et) & (events_df['line_fraction'] == str(frac)) & (~events_df['is_retro'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(subset['fwd_mfe_10'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', label='Breakeven')
axes[0].axvline(subset['fwd_mfe_10'].mean(), color='green', linestyle='-', label=f'Mean: {subset["fwd_mfe_10"].mean():.2f}%')
axes[0].set_xlabel('MFE at 10 bars (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'{et} on {frac} line - MFE Distribution')
axes[0].legend()

win_by_frac = events_df[(events_df['event_type'] == et) & (~events_df['is_retro'])].groupby('line_fraction')['fwd_win_10'].apply(lambda x: x.astype(bool).mean()).sort_values()
win_by_frac.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Win Rate')
axes[1].set_title(f'{et} Win Rate by Line Fraction')
axes[1].axvline(0.5, color='red', linestyle='--', label='50% threshold')
axes[1].legend()

plt.tight_layout()
plt.show()